# Week 12: Deployment Validation

This notebook validates the deployed API through its SSH tunnel before public release. It reuses the frozen Week 11 evaluation set to compare search quality, cache-aware latency, detail retrieval, and the bounded quality-rerank path.

---

In [ ]:
import json
import math
import os
import time
import uuid
from collections import defaultdict
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 180)

API_BASE_URL = os.getenv("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
TOP_K = 5
PROFILES = ["fast", "balanced", "quality"]
PROFILE_ORDER = pd.CategoricalDtype(categories=PROFILES, ordered=True)
REQUEST_PAUSE_SECONDS = 0.9
REQUEST_TIMEOUT_SECONDS = 60

REQUEST_HEADERS = {"X-Search-Session-ID": uuid.uuid4().hex}

session = requests.Session()
session.headers.update(REQUEST_HEADERS)

## 1. Deployment Artifacts and API Readiness

Start an SSH tunnel before running this notebook: `ssh -L 8000:127.0.0.1:8000 <vm>`. The relevance labels were frozen against the pre-demo snapshot. This section verifies that the deployed API still serves the same retrieval population and model settings before reporting benchmark results.

---

In [ ]:
query_dataset = json.loads(Path("../data/processed/search_relevance_queries.json").read_text())
final_manifest = json.loads(Path("../data/processed/search_relevance_final_manifest.json").read_text())
qrels = [
    json.loads(line)
    for line in Path("../data/processed/search_relevance_final_qrels.jsonl").read_text().splitlines()
    if line
]

query_items = query_dataset["items"]
test_items = [item for item in query_items if item["split"] == "test"]

ready_response = session.get(f"{API_BASE_URL}/ready", timeout=REQUEST_TIMEOUT_SECONDS)
ready_response.raise_for_status()
ready = ready_response.json()

active_pointer = json.loads(Path("../data/models/search/active.json").read_text())
active_snapshot_dir = Path(active_pointer["snapshot_path"])
active_manifest = json.loads((active_snapshot_dir / "manifest.json").read_text())
frozen_snapshot_dir = Path("../data/models/search_snapshots") / final_manifest["snapshot_id"]
frozen_snapshot_manifest = json.loads((frozen_snapshot_dir / "manifest.json").read_text())

compatibility_fields = [
    "public_listing_count",
    "retrievable_listing_count",
    "compliance_rule_version",
    "dense_model",
]
compatibility = pd.DataFrame([
    {
        "field": field,
        "frozen benchmark": frozen_snapshot_manifest[field],
        "active API": active_manifest[field],
        "matches": frozen_snapshot_manifest[field] == active_manifest[field],
    }
    for field in compatibility_fields
])

assert ready["snapshot_id"] == active_manifest["snapshot_id"]
assert compatibility["matches"].all()

display(pd.DataFrame([
    {
        "api_base_url": API_BASE_URL,
        "active_snapshot": ready["snapshot_id"],
        "frozen_benchmark_snapshot": final_manifest["snapshot_id"],
        "benchmark_queries": len(query_items),
        "held_out_test_queries": len(test_items),
        "qrels": len(qrels),
    }
]))
compatibility

## 2. Live API Benchmark

Each query/profile pair is sent once, then repeated immediately with the same payload. This produces an actual cache miss and cache hit for every pair. The request pace stays below the production rate limit, and the notebook uses a separate session ID so it does not share a rate-limit bucket with the browser.

---

In [ ]:
def post_json(path, payload):
    for attempt in range(2):
        started_at = time.perf_counter()
        response = session.post(
            f"{API_BASE_URL}{path}",
            json=payload,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        elapsed_ms = (time.perf_counter() - started_at) * 1000
        if response.status_code == 429 and attempt == 0:
            time.sleep(float(response.headers.get("Retry-After", 1)))
            continue
        response.raise_for_status()
        return response.json(), elapsed_ms, response.headers
    raise RuntimeError("Request remained rate limited after one retry.")


def search_request(item, profile):
    body, elapsed_ms, headers = post_json(
        "/search",
        {
            "query": item["query"],
            "top_k": TOP_K,
            "sort_by": None,
            "search_profile": profile,
        },
    )
    return body, {
        "query_id": item["id"],
        "split": item["split"],
        "category": item["category"],
        "profile": profile,
        "http_status": 200,
        "cache_status": headers.get("X-Cache", "BYPASS"),
        "observed_latency_ms": round(elapsed_ms, 2),
        "reported_compute_latency_ms": body["meta"]["timings_ms"].get("total"),
        "cross_encoder_rerank_ms": body["meta"]["timings_ms"].get("cross_encoder_rerank"),
        "effective_profile": body["meta"]["effective_profile"],
        "reranker_used": body["meta"]["reranker_used"],
        "returned": len(body["results"]),
        "result_ids": [str(row["listing_id"]) for row in body["results"]],
    }


response_by_key = {}
benchmark_rows = []

for profile in PROFILES:
    for item in query_items:
        for phase in ["initial", "repeat"]:
            body, row = search_request(item, profile)
            row["phase"] = phase
            benchmark_rows.append(row)
            response_by_key[phase, profile, item["id"]] = body
            time.sleep(REQUEST_PAUSE_SECONDS)

benchmark = pd.DataFrame(benchmark_rows)
benchmark["profile"] = benchmark["profile"].astype(PROFILE_ORDER)
assert (benchmark["http_status"] == 200).all()
assert (benchmark["returned"] == TOP_K).all()

execution_summary = benchmark.groupby(
    ["phase", "profile", "cache_status"],
    as_index=False,
    observed=True,
).agg(
    requests=("query_id", "count"),
    returned_results=("returned", "sum"),
)
execution_summary

## 3. Held-Out Search Quality

The final quality score uses only the 12 held-out queries. A result is relevant at grade 2 or above; NDCG keeps the full four-level relevance scale. Judgment coverage is checked before any score is reported.

---

In [ ]:
RELEVANT_GRADE = 2
qrel_by_key = {
    (row["query_id"], str(row["listing_id"])): row["relevance_grade"]
    for row in qrels
}
grades_by_query = defaultdict(list)
for row in qrels:
    grades_by_query[row["query_id"]].append(row["relevance_grade"])


def precision_at_five(grades):
    return sum(grade >= RELEVANT_GRADE for grade in grades) / TOP_K


def mrr_at_five(grades):
    for rank, grade in enumerate(grades, start=1):
        if grade >= RELEVANT_GRADE:
            return 1 / rank
    return 0.0


def ndcg_at_five(grades, ideal_grades):
    def dcg(values):
        return sum((2 ** grade - 1) / math.log2(rank + 1) for rank, grade in enumerate(values, start=1))

    ideal = dcg(sorted(ideal_grades, reverse=True)[:TOP_K])
    return dcg(grades) / ideal if ideal else 0.0


quality_rows = []
for profile in PROFILES:
    for item in test_items:
        result = response_by_key["initial", profile, item["id"]]
        result_ids = [str(row["listing_id"]) for row in result["results"]]
        missing = [listing_id for listing_id in result_ids if (item["id"], listing_id) not in qrel_by_key]
        quality_rows.append({
            "query_id": item["id"],
            "profile": profile,
            "judged_top_5": TOP_K - len(missing),
            "missing_judgments": missing,
            "grades": [qrel_by_key.get((item["id"], listing_id)) for listing_id in result_ids],
        })

quality_by_query = pd.DataFrame(quality_rows)
quality_by_query["profile"] = quality_by_query["profile"].astype(PROFILE_ORDER)
assert (quality_by_query["judged_top_5"] == TOP_K).all()

quality_by_query["precision_at_5"] = quality_by_query["grades"].apply(precision_at_five)
quality_by_query["mrr_at_5"] = quality_by_query["grades"].apply(mrr_at_five)
quality_by_query["ndcg_at_5"] = quality_by_query.apply(
    lambda row: ndcg_at_five(row["grades"], grades_by_query[row["query_id"]]),
    axis=1,
)

quality_summary = quality_by_query.groupby("profile", as_index=False, observed=True).agg(
    held_out_queries=("query_id", "count"),
    judged_top_5_coverage=("judged_top_5", lambda values: values.sum() / (len(values) * TOP_K)),
    precision_at_5=("precision_at_5", "mean"),
    ndcg_at_5=("ndcg_at_5", "mean"),
    mrr_at_5=("mrr_at_5", "mean"),
)
quality_summary.round(3)

### 3.1 Hard-Filter Integrity

Relevance should not come at the cost of explicit requirements. The held-out set contains city, maximum-price, and minimum-bedroom constraints, which can be checked directly against the public result fields.


In [ ]:
def meets_hard_requirements(result, requirements):
    if "city" in requirements and str(result.get("city", "")).casefold() != requirements["city"].casefold():
        return False
    if "price_max" in requirements and (result.get("price") is None or result["price"] > requirements["price_max"]):
        return False
    if "beds_min" in requirements and (result.get("beds") is None or result["beds"] < requirements["beds_min"]):
        return False
    return True


filter_rows = []
for profile in PROFILES:
    for item in test_items:
        requirements = item["hard_requirements"]
        result = response_by_key["initial", profile, item["id"]]
        matches = [meets_hard_requirements(row, requirements) for row in result["results"]]
        filter_rows.append({
            "query_id": item["id"],
            "profile": profile,
            "has_explicit_filter": bool(requirements),
            "valid_results": sum(matches),
            "returned_results": len(matches),
            "query_fully_valid": all(matches),
        })

filter_integrity = pd.DataFrame(filter_rows)
filter_integrity["profile"] = filter_integrity["profile"].astype(PROFILE_ORDER)
filter_integrity.groupby("profile", as_index=False, observed=True).agg(
    held_out_queries=("query_id", "count"),
    queries_with_explicit_filters=("has_explicit_filter", "sum"),
    result_constraint_accuracy=("valid_results", "sum"),
    total_results=("returned_results", "sum"),
    fully_valid_queries=("query_fully_valid", "sum"),
).assign(
    result_constraint_accuracy=lambda frame: frame["result_constraint_accuracy"] / frame["total_results"],
    fully_valid_query_rate=lambda frame: frame["fully_valid_queries"] / frame["held_out_queries"],
).round(3)

## 4. Cache-Aware Latency

Observed latency is the client-visible HTTP time. API compute latency and Cross Encoder timing are interpreted only for cache misses because cached responses retain the original search metadata. The tables group by the actual cache header instead of assuming the initial pass was cold.

---

In [ ]:
def latency_summary(frame, column):
    values = frame[column].dropna()
    return pd.Series({
        "requests": len(values),
        "p50_ms": values.quantile(0.50),
        "p90_ms": values.quantile(0.90),
        "p95_ms": values.quantile(0.95),
    })


http_latency = (
    benchmark.groupby(
        ["profile", "phase", "cache_status"],
        group_keys=False,
        observed=True,
    )
    .apply(lambda frame: latency_summary(frame, "observed_latency_ms"))
    .reset_index()
    .sort_values(["profile", "phase", "cache_status"])
)

api_compute_latency = (
    benchmark.query("phase == 'initial' and cache_status == 'MISS'")
    .groupby("profile", group_keys=False, observed=True)
    .apply(lambda frame: latency_summary(frame, "reported_compute_latency_ms"))
    .reset_index()
    .sort_values("profile")
)

display(http_latency.round(2))
display(api_compute_latency.round(2))

rerank_latency = (
    benchmark.query("profile == 'quality' and phase == 'initial' and cache_status == 'MISS'")
    .pipe(lambda frame: latency_summary(frame, "cross_encoder_rerank_ms"))
    .to_frame().T
)
rerank_latency.round(2)

## 5. Listing Details Flow

The product page loads details in batches after search results are available. This checks the same `/listings/details` path for every held-out Quality result page and measures its miss/hit behavior.

---

In [ ]:
detail_rows = []
details_by_query = {}

for phase in ["initial", "repeat"]:
    for item in test_items:
        search_result = response_by_key["initial", "quality", item["id"]]
        listing_ids = [str(row["listing_id"]) for row in search_result["results"]]
        body, elapsed_ms, headers = post_json("/listings/details", {"listing_ids": listing_ids})
        details = body["listings"]
        detail_rows.append({
            "query_id": item["id"],
            "phase": phase,
            "cache_status": headers.get("X-Cache", "BYPASS"),
            "observed_latency_ms": round(elapsed_ms, 2),
            "requested": len(listing_ids),
            "returned": len(details),
            "ids_match": [str(row["listing_id"]) for row in details] == listing_ids,
            "descriptions_present": sum(bool(row.get("listing_description")) for row in details),
        })
        details_by_query[phase, item["id"]] = {str(row["listing_id"]): row for row in details}
        time.sleep(REQUEST_PAUSE_SECONDS)

detail_benchmark = pd.DataFrame(detail_rows)
assert detail_benchmark["ids_match"].all()
assert (detail_benchmark["descriptions_present"] == TOP_K).all()

detail_integrity = detail_benchmark.groupby("phase", as_index=False).agg(
    batches=("query_id", "count"),
    cache_states=("cache_status", lambda values: ", ".join(sorted(values.unique()))),
    requested_listings=("requested", "sum"),
    returned_listings=("returned", "sum"),
    descriptions_present=("descriptions_present", "sum"),
)

detail_latency = (
    detail_benchmark.groupby(["phase", "cache_status"], group_keys=False)
    .apply(lambda frame: latency_summary(frame, "observed_latency_ms"))
    .reset_index()
)

display(detail_integrity)
detail_latency.round(2)

## 6. Product Examples

These examples use the default `quality` profile and show the response fields that feed its search cards: structured facts, summaries, matched preferences, and the original listing description returned by the batch-details endpoint.

---

In [ ]:
example_ids = ["rel_029", "rel_032", "rel_039"]
example_rows = []

for query_id in example_ids:
    item = next(item for item in test_items if item["id"] == query_id)
    result = response_by_key["initial", "quality", query_id]
    details = details_by_query["initial", query_id]
    for row in result["results"][:3]:
        detail = details[str(row["listing_id"])]
        example_rows.append({
            "query": item["query"],
            "rank": row["rank"],
            "address": row["address"],
            "city": row["city"],
            "price": row["price"],
            "beds": row["beds"],
            "baths": row["baths"],
            "sqft": row["sqft"],
            "matched_preferences": ", ".join(match["value"] for match in row["matched_signals"]),
            "summary": row["summary"],
            "description_excerpt": detail["listing_description"][:260],
        })

pd.DataFrame(example_rows)

## 7. Runtime Metrics Snapshot

The dashboard aggregates anonymous interactions recorded by the Streamlit application. It is operational context rather than a controlled benchmark, so this notebook does not add events to it. Set `API_DEMO_METRICS_TOKEN` in the terminal before starting Jupyter when access to this admin-only endpoint is required.

---

In [ ]:
metrics_token = os.getenv("API_DEMO_METRICS_TOKEN", "")
metrics_headers = {"X-Demo-Metrics-Token": metrics_token} if metrics_token else {}
metrics_response = session.get(
    f"{API_BASE_URL}/demo/metrics",
    headers=metrics_headers,
    timeout=REQUEST_TIMEOUT_SECONDS,
)

if metrics_response.ok:
    runtime_metrics = metrics_response.json()
    display(pd.DataFrame([
        {
            "searches": runtime_metrics["query_volume"],
            "sessions": runtime_metrics["unique_sessions"],
            "profile_comparisons": runtime_metrics["comparison_searches"],
            "zero_result_rate": runtime_metrics["zero_result_rate"],
            "feedback_responses": runtime_metrics["satisfaction"]["responses"],
            "helpful_rate": runtime_metrics["satisfaction"]["helpful_rate"],
        }
    ]))
    pd.DataFrame([
        {
            "profile": profile,
            "searches": runtime_metrics["profile_usage"][profile],
            "api_p50_ms": runtime_metrics["profile_latency_ms"][profile]["api"]["p50"],
            "api_p90_ms": runtime_metrics["profile_latency_ms"][profile]["api"]["p90"],
            "api_p95_ms": runtime_metrics["profile_latency_ms"][profile]["api"]["p95"],
        }
        for profile in PROFILES
    ])
else:
    print(f"Runtime metrics unavailable: HTTP {metrics_response.status_code}.")

## 8. Quality Rerank Queue

The deployed service permits one Cross Encoder execution at a time. These distinct cache-miss probes are sent together to measure queueing directly. Results are ordered by completion time, not input order.

---

In [ ]:
from concurrent.futures import ThreadPoolExecutor


quality_probes = [
    "three bedroom home in Irvine with a fireplace",
    "three bedroom home in Irvine with a fireplace please",
    "three bedroom home in Irvine with a fireplace today",
]


def quality_probe(query):
    started_at = time.perf_counter()
    response = requests.post(
        f"{API_BASE_URL}/search",
        json={"query": query, "top_k": TOP_K, "search_profile": "quality"},
        timeout=REQUEST_TIMEOUT_SECONDS,
        headers=REQUEST_HEADERS,
    )
    elapsed_ms = round((time.perf_counter() - started_at) * 1000, 2)
    body = response.json()
    return {
        "query": query,
        "http_status": response.status_code,
        "cache_status": response.headers.get("X-Cache", "BYPASS"),
        "observed_latency_ms": elapsed_ms,
        "completed_after_start_ms": round((time.perf_counter() - queue_started_at) * 1000, 2),
        "reranker_used": body.get("meta", {}).get("reranker_used"),
        "error_code": body.get("error", {}).get("code"),
    }


queue_started_at = time.perf_counter()

with ThreadPoolExecutor(max_workers=len(quality_probes)) as executor:
    queue_results = list(executor.map(quality_probe, quality_probes))

queue_results = pd.DataFrame(queue_results)
assert (queue_results["http_status"] == 200).all()
assert queue_results["reranker_used"].all()
queue_results.sort_values("completed_after_start_ms").reset_index(drop=True)

## 9. Review Notes

- The held-out table is the product relevance result; the broader 40-query run is used for latency only.
- Each query/profile pair is issued twice in immediate succession, so the cache comparison reflects the same payload within its TTL.
- Quality cache misses include Cross Encoder timing; queue probes intentionally submit concurrent cache misses and should be interpreted separately.
- The details check covers the same batch endpoint used by the product page, including original listing descriptions.
- Runtime metrics reflect local demo usage and should not be compared directly with the controlled benchmark.